# From notebooks to a full pipeline: [popgen-npe](https://github.com/kr-colab/popgen-npe)

In this final notebook we shift from hand-rolled NPE on toy demographies to a research-grade pipeline. We will walk through the [popgen-npe Soup-to-Nuts tutorial](https://popgen-npe.readthedocs.io/en/latest/tutorial.html) — a six-step recipe for inferring a **recombination rate landscape** along a chromosome — and run the prediction step live against a pre-trained checkpoint shipped with this repository.

> **What you'll do:** inspect the simulator, processor, and YAML config that define the run; understand what `training_workflow.smk` would do (we don't retrain in-session); run `prediction_workflow.smk` against an example VCF; load the resulting per-window posterior samples and plot the inferred rate landscape.

## What we just practiced
- Defined priors using the framework provided by `sbi` over demographic parameters (Ne, recombination) and simulators (`msprime`) to generate training data $\theta$. 
- Built summary statistics (AFS) as training data $x$. So that we have parameter pairs $(\theta, x)$.
- Trained NPE to recover posteriors $p(\theta \mid x_o)$ given observation $x_o$.
- Explored calibration/fit checks (PPC, SBC) and the effect of priors and simulator choices.
- Saw how demographic complexity (multiple epochs, recombination, etc.) reshapes the summaries we pass into NPE.

All in all, we now have a general knowledge about the `sbi` package and `msprime` simulation.

## Why a workflow?
Interactive notebooks are great for intuition and prototyping, but real projects often need reproducible, scalable runs: consistent environments, large simulation budgets, checkpoints, and logs.

A workflow system (Snakemake) coordinates simulations, training, diagnostics, and plotting so you can rerun or extend experiments without manual steps.

## What popgen-npe provides
- A Snakemake pipeline that wires together simulation, summary stats extraction, NPE training, and diagnostics.
- Priors and simulators defined in configs (YAML files) so you can swap demographic models, summary stats, or density estimators without editing code.
- Resource-aware execution: run locally or on clusters/SLURM with the same recipes.
- Logging and checkpoints: simulation caches, training logs, and reproducible outputs.

## What we are building: a recombination rate landscape

The Soup-to-Nuts tutorial trains an NPE that, given the genotype matrix of a genomic window, returns a posterior over the local recombination rate $r$. Sliding that estimator along a chromosome yields a **rate landscape** — a per-window posterior that we can summarise (e.g. by its mean) and plot against genomic position.

Conceptually the pipeline is:

```
parameter prior (BoxUniform over r)
       │
       ▼
msprime simulator  ──►  tree sequence  ──►  processor (genotype tensor)
                                                       │
                                                       ▼
                                          embedding network + normalising flow
                                                       │
                                                       ▼
                                            posterior  p(r | x_window)
```

The six steps below mirror the tutorial. Steps 1–4 are configuration we set up *before* the workshop; Step 5 runs live on a small example VCF; Step 6 visualises the result.

## Step 1 — Write the simulator

popgen-npe ships several simulators (`recombination_rate`, `VariablePopulationSize`, `AraTha_2epoch`, …) and lets you subclass `BaseSimulator` to add new ones. For rate-landscape inference the tutorial uses a single-population constant-size model with $r$ as the only inferred parameter.

Below is the simulator class used by the tutorial. We're not editing it here — it lives inside the installed `popgen-npe` package — but it's worth reading once: every field in `default_config` is something the YAML in Step 3 is allowed to override.

In [ ]:
# TODO: paste the simulator class from
# https://popgen-npe.readthedocs.io/en/latest/tutorial.html#step-1-write-the-simulator
#
# Expected shape (from the surrounding API docs):
#   class RateLandscapeSimulator(BaseSimulator):
#       default_config = {
#           "samples": {...},
#           "sequence_length": ...,
#           "mutation_rate": ...,
#           "pop_size": ...,
#           "recombination_rate": [lo, hi],   # the prior
#       }
#       def __init__(self, config): ...
#       def __call__(self, seed=None):
#           ...
#           return ts, theta


## Step 2 — Configure the processor

Processors turn tree sequences into the tensors the embedding network consumes. For rate inference, the tutorial picks a processor that produces a genotype matrix with positional information (so the network can pick up LD-based signal). The processor's `default_config` lists every knob — `n_snps`, `maf_thresh`, `phased`, etc. — and the YAML in Step 3 only overrides the ones we care about.

In [ ]:
# TODO: paste the processor block from
# https://popgen-npe.readthedocs.io/en/latest/tutorial.html#step-2-configure-the-processor
#
# Expected: a small YAML snippet or Python dict, e.g.
#   processor:
#       class_name: cnn_extract   # or ReLERNN_processor
#       n_snps: ...
#       maf_thresh: ...
#       phased: ...


## Step 3 — Write the config YAML

The full run is described by one YAML file. It collects the simulator and processor blocks above plus training hyperparameters, resource limits, and (for prediction) the VCF + BED-window inputs. The same file is consumed by both `training_workflow.smk` and `prediction_workflow.smk`, which is what guarantees that prediction-time tree sequences are processed identically to training-time ones.

The config we'll use in this session is shipped at `popgen_npe_demo/config.yaml` alongside its pre-trained checkpoint. We print it here so you can see exactly what was used.

In [ ]:
# TODO: paste the full config YAML from
# https://popgen-npe.readthedocs.io/en/latest/tutorial.html#step-3-write-the-config-yaml
# into popgen_npe_demo/config.yaml, and uncomment the print below.
#
# from pathlib import Path
# CONFIG_PATH = Path('popgen_npe_demo/config.yaml')
# print(CONFIG_PATH.read_text())


## Step 4 — Run the training workflow *(pre-run for this session)*

In a real project you would now run:

```bash
snakemake --cores N \
          --configfile popgen_npe_demo/config.yaml \
          --snakefile workflow/training_workflow.smk
```

This triggers the full simulate → process → train pipeline: parameter draws from the prior, msprime simulations chunked across `n_chunk` workers, feature extraction into a Zarr store, then training of the embedding network plus normalising flow with the hyperparameters from the YAML.

**We do not run this live.** A meaningful training run takes minutes to hours and benefits from a GPU. Instead, this repository ships the resulting checkpoint at `popgen_npe_demo/checkpoint/`, so Step 5 can load it directly.

In [ ]:
# Reference only — do not run during the session.
# Uncomment locally if you want to reproduce the training run from scratch.
#
# import subprocess
# subprocess.run([
#     "snakemake",
#     "--cores", "4",
#     "--configfile", "popgen_npe_demo/config.yaml",
#     "--snakefile", "workflow/training_workflow.smk",
# ], check=True)


## Step 5 — Run prediction on the example data *(live)*

Prediction reuses the simulator, processor, and embedding network sections from the same YAML. Tree sequences are now *inferred from a VCF* rather than simulated, then processed and scored by the trained flow — yielding a posterior per genomic window.

The cell below runs `prediction_workflow.smk` against the example VCF shipped in `popgen_npe_demo/example_vcf/`. If the checkpoint or example data are missing, it falls back to a no-op so the notebook still executes cleanly.

In [ ]:
import subprocess
from pathlib import Path

DEMO_DIR = Path('popgen_npe_demo')
CONFIG = DEMO_DIR / 'config.yaml'
CHECKPOINT = DEMO_DIR / 'checkpoint'
VCF = DEMO_DIR / 'example_vcf'

# Path to a clone of https://github.com/kr-colab/popgen-npe with
# workflow/prediction_workflow.smk inside.
POPGEN_NPE_DIR = Path('../../popgen-npe')   # adjust if needed

missing = [p for p in (CONFIG, CHECKPOINT, VCF, POPGEN_NPE_DIR) if not p.exists()]
if missing:
    print('Skipping live prediction — missing:', *missing, sep='\n  - ')
    print('\nGenerate these by running training once (Step 4) and cloning popgen-npe.')
else:
    subprocess.run([
        'snakemake',
        '--cores', '2',
        '--configfile', str(CONFIG.resolve()),
        '--snakefile', str((POPGEN_NPE_DIR / 'workflow' / 'prediction_workflow.smk').resolve()),
    ], check=True)


## Step 6 — Interpret the results: the rate landscape

The prediction workflow writes one posterior per window. We load those, summarise each window by its posterior mean (and a credible interval), and plot the resulting landscape against genomic position. The expected output for this exact config is cached at `popgen_npe_demo/expected_outputs/rate_landscape.png` — useful as a fallback if the live run was skipped.

In [ ]:
# TODO: load per-window posterior samples produced by prediction_workflow.smk
# and plot the rate landscape.
#
# Sketch:
#   import numpy as np, matplotlib.pyplot as plt
#   posteriors = np.load('popgen_npe_demo/checkpoint/predictions.npz')
#   # x-axis: window midpoints from the BED file
#   # y-axis: posterior mean recombination rate per window
#   # shaded band: 5th-95th percentile
#
# If the live prediction was skipped, display the cached figure:
#   from IPython.display import Image, display
#   display(Image('popgen_npe_demo/expected_outputs/rate_landscape.png'))


## Next steps and scaling to real data

Once you've reproduced the demo, the natural extensions are:

- **Swap the simulator** for a model that matches your study system (e.g., a two-epoch demography, or a multi-population split). See `simulators.html` for the menu and the "Creating Custom Simulators" recipe.
- **Swap the processor** to change the summary the network sees — `tskit_sfs` for an SFS-based run, `tskit_windowed_sfs_plus_ld` to add LD information, `cnn_extract` for raw genotype matrices.
- **Scale up training** by raising `n_train` and pushing the run to a cluster: Snakemake's SLURM profile is configured directly in the YAML's `gpu_resources` and `cpu_resources` blocks.
- **Validate before interpreting** — re-use the PPC and SBC machinery from notebooks 3 and 4 against held-out posterior draws before treating the rate landscape as ground truth.
- **Contribute back.** New simulators, processors, and bug fixes are welcome via the [popgen-npe contributor guide](https://popgen-npe.readthedocs.io/en/latest/contributing.html).

That's the bridge from the tutorial notebooks to a real-world NPE pipeline. Explore the popgen-npe repo and configs to continue experimenting beyond this session.